# Week 23: Next-Gen AI Reasoning, MCTS & Test-Time Compute
## Notebook 01: Monte Carlo Tree Search (MCTS) from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In complex problem solving (Go, Chess, complex mathematical theorem proving, OpenAI o1):
- Greedy next-token prediction frequently makes irreversible early missteps.
- Exhaustive brute-force search over all possibilities suffers from combinatorial explosion ($B^D$).

**Monte Carlo Tree Search (MCTS)** balances exploration and exploitation using the **Upper Confidence Bound (UCB1)**:
1. **Selection:** Traverse down the most promising tree branch.
2. **Expansion:** Create a new candidate child state.
3. **Simulation (Rollout):** Run a fast heuristic or random evaluation to the end.
4. **Backpropagation:** Update win/value rates upward along the entire path!

In [ ]:
import math
import numpy as np

# Step 1: Implement MCTS Node
class MCTSNode:
    def __init__(self, state, parent=None):
        self.state = state          # e.g., current value
        self.parent = parent
        self.children = []
        self.visits = 0
        self.value_sum = 0.0

    def is_fully_expanded(self, possible_actions):
        return len(self.children) == len(possible_actions)

    def ucb1(self, exploration_weight=1.414):
        if self.visits == 0:
            return float("inf")
        # Exploitation + Exploration
        exploitation = self.value_sum / self.visits
        exploration = exploration_weight * math.sqrt(math.log(self.parent.visits) / self.visits)
        return exploitation + exploration

# Step 2: Implement MCTS Solver for a Simple Target Reach Game (Target = 20)
# Actions: +2, +3, +5
target_goal = 20
actions = [2, 3, 5]

root = MCTSNode(state=0)

def select(node):
    while node.children:
        # Choose child with highest UCB1
        node = max(node.children, key=lambda c: c.ucb1())
    return node

def expand(node):
    # Find untried action
    tried_states = [c.state for c in node.children]
    for a in actions:
        next_s = node.state + a
        if next_s not in tried_states and next_s <= target_goal:
            child = MCTSNode(state=next_s, parent=node)
            node.children.append(child)
            return child
    return node

def simulate(node):
    # Random rollout to terminal state
    curr = node.state
    while curr < target_goal:
        curr += np.random.choice(actions)
    # Reward: 1.0 if exactly hit target, 0.0 if overshot
    return 1.0 if curr == target_goal else 0.0

def backpropagate(node, reward):
    while node is not None:
        node.visits += 1
        node.value_sum += reward
        node = node.parent

# Step 3: Run 500 MCTS Iterations
for _ in range(500):
    leaf = select(root)
    child = expand(leaf)
    reward = simulate(child)
    backpropagate(child, reward)

# Best Move from Root
best_child = max(root.children, key=lambda c: c.visits)

print("--- MCTS Tree Search Optimization ---")
print(f"Goal: Reach exactly {target_goal} from 0 using steps [2, 3, 5]")
print(f"Total Root Visits: {root.visits}\n")
print("Candidate First Moves Evaluated by MCTS:")
for c in root.children:
    win_rate = (c.value_sum / c.visits) * 100
    print(f"  Action +{c.state}: Visits = {c.visits:>3} | Success Rate = {win_rate:>5.1f}% | UCB1 = {c.ucb1():.3f}")

print(f"\nOptimal First Move Selected by MCTS: +{best_child.state} (Most Visited)")

### 3. Step-by-Step Code Walkthrough
1. **UCB1 Balance:** Balances highly rewarding moves with less-visited moves.
2. **Selective Tree Expansion:** Nodes with high success rates receive more computational simulations.
3. **Robust Move Selection:** The best move is chosen by visit count (most robust), not just single lucky rollouts.

---

### 4. Industry Applications & Trade-offs
- **AlphaGo / AlphaZero:** Mastered Go, Chess, and Shogi defeating world champions.
- **Modern Reasoning LLMs:** Using MCTS to search over chain-of-thought candidate reasoning branches.

---

### 5. Top Interview Questions & Answers
**Q1: Why is UCB1 exploration logarithmic ($\sqrt{\frac{\ln N}{n}}$)?**
> **Answer:** As total parent visits $N$ grow, the exploration bonus grows slowly (logarithmically), ensuring the algorithm eventually exploits the best known path while never permanently abandoning unvisited branches.

**Q2: How does AlphaZero replace the random rollout simulation phase of classic MCTS?**
> **Answer:** Classic MCTS runs hundreds of random steps to the end of the game (noisy and slow). AlphaZero truncates the rollout immediately at the new child node and queries a trained **Value Neural Network** $V(s) \in [-1, +1]$ to evaluate position quality in a single forward pass.